# Can VirtualiZarr help us access NISAR data?

NISAR standard products are HDF5 files in S3, tens of GB each. The usual way to build a time
series is to open every granule and let h5py/xarray walk each file's internal metadata — which
is slow, and which you pay for again on every re-open.

[VirtualiZarr](https://virtualizarr.readthedocs.io/) offers a different deal: read each file's
chunk offsets *once*, store them as a Zarr manifest, and afterwards treat the whole archive as a
single Zarr array. **No pixels are copied** — reads still stream from the original `.h5` files in
the ASF bucket. Only the *index* is new.

This notebook tests whether that actually works on real NISAR granules, and where it breaks.

## What we found

| | Result |
|---|---|
| Virtualize one real GCOV granule | works — 0.85 s |
| Baseline: same granule via `h5netcdf` + `s3fs` | 17 s |
| Concatenate a 24-date time series | works — 0.56 TB logical cube |
| Persist the cube to Icechunk | works — **13 MB** index on disk |
| Re-open the persisted cube | 0.05 s |
| Pixels byte-identical to the source HDF5 | yes |
| Parse a granule *without* scoping to a group | **fails** — complex-valued attributes |
| Concatenate all 29 granules on the track | **fails** — 2 different output grids |
| GSLC | works |
| RSLC | works only after dropping non-raster variables |

The headline: **VirtualiZarr removes the metadata cost of opening NISAR archives, not the pixel
cost.** That is worth a lot for repeat-visit time series, and nothing at all for a single-granule
read.

## Setup

Requires a `~/.netrc` with Earthdata Login credentials, and — for direct S3 access —
a machine in `us-west-2`. Helpers live in `nisar_virtual.py` next to this notebook.

In [ ]:
import sys, time, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import xarray as xr
import h5py
import virtualizarr as vz
from virtualizarr.parsers import HDFParser
from obspec_utils.readers import BlockStoreReader

sys.path.insert(0, ".")
import nisar_virtual as nv

# zarr warns that fixed-length byte dtypes (NISAR's `listOfPolarizations` etc.)
# have no stable Zarr v3 spec yet. Noted, not fatal.
warnings.filterwarnings("ignore", message=".*does not have a Zarr V3 specification.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="earthaccess.*")

# Section 4 deliberately triggers a failure inside zarr's async group listing;
# without this, the orphaned tasks print tracebacks over later cells.
import asyncio
asyncio.get_event_loop().set_exception_handler(lambda loop, ctx: None)

BBOX = (-121.932, 46.754, -121.5707, 46.964)  # Mt. Rainier
registry = nv.obstore_registry()   # authenticates via ~/.netrc, returns temporary S3 creds
print("group of interest:", nv.GCOV_GRIDS)

## 1. Find a repeat-pass stack

VirtualiZarr can only concatenate granules that already share a pixel grid — it cannot reproject
or resample. For NISAR that means grouping by **relative orbit and frame** before anything else.

In [ ]:
track = nv.find_track(BBOX)
for (direction, relorb, frame, pols), items in track.items():
    print(f"{direction} rel.orbit {relorb} frame {frame} {pols}: "
          f"{len(items):2d} granules, {items[0][0][:8]} -> {items[-1][0][:8]}")

key, items = next(iter(track.items()))   # the deepest stack

## 2. Virtualize a single granule

`HDFParser(group=...)` reads the file's chunk index and returns an xarray Dataset whose arrays are
`ManifestArray`s — byte offsets into the original HDF5, not data.

Scoping to a group is **mandatory**, for reasons shown in section 4.

In [ ]:
parser = HDFParser(group=nv.GCOV_GRIDS)

t0 = time.perf_counter()
vds = vz.open_virtual_dataset(items[0][1], registry=registry, parser=parser)
t_one = time.perf_counter() - t0
print(f"manifest built in {t_one:.2f}s\n")
vds

In [ ]:
# What the parser recovered from the HDF5 layout:
for name in ("HHHH", "mask"):
    a = vds[name].data
    print(f"{name}")
    print(f"   shape   {a.shape}")
    print(f"   chunks  {a.metadata.chunks}")
    print(f"   dtype   {a.metadata.data_type}")
    print(f"   codecs  {a.metadata.codecs}")
    print(f"   {len(a.manifest):,} chunk references\n")

NISAR's HDF5 filters (`shuffle` + `deflate`) map cleanly onto Zarr v3 codecs, which is the whole
reason this is possible: Zarr can decompress NISAR's chunks *in place*, without rewriting them.

Note the float32 bands use `zlib(level=1)` while the uint8 masks use `zlib(level=9)` — the parser
picks this up per-variable.

## 3. What does this save? Compare against the obvious approach

The status-quo way to open a NISAR granule lazily is `xarray` + `h5netcdf` over `s3fs`.

In [ ]:
import s3fs
c = nv.nisar_s3_credentials()
fs = s3fs.S3FileSystem(key=c["accessKeyId"], secret=c["secretAccessKey"],
                       token=c["sessionToken"])

t0 = time.perf_counter()
ds_baseline = xr.open_dataset(fs.open(items[0][1].replace("s3://", ""), "rb"),
                              engine="h5netcdf", group=nv.GCOV_GRIDS, phony_dims="access")
t_baseline = time.perf_counter() - t0

print(f"h5netcdf + s3fs lazy open : {t_baseline:5.1f}s")
print(f"VirtualiZarr manifest     : {t_one:5.2f}s   ({t_baseline / t_one:.0f}x faster)")

Two caveats on that number, so nobody over-reads it:

1. Part of the gain is the *reader*, not VirtualiZarr. Its `HDFParser` reads through obstore's
   `BlockStoreReader`, which batches range requests far better than default-configured `s3fs`.
   A tuned `s3fs` blockcache narrows this gap considerably.
2. The manifest is doing less work: it records chunk offsets and does not build CF-decoded
   xarray objects.

The durable win is not this first pass at all — it is that the manifest can be **saved**, so the
second open costs nothing (section 7).

## 4. Failure: you cannot parse a NISAR granule at the root

The natural first thing to try is to point the parser at the whole file. It fails.

In [ ]:
try:
    vz.open_virtual_datatree(items[0][1], registry=registry, parser=HDFParser())
except Exception as e:
    print(f"{type(e).__name__}: {e}")

Where does a `complex` come from? Not from the imagery — from *attributes* in the metadata tree:

In [ ]:
hits = []

def collect_complex(name, obj):
    for k, v in obj.attrs.items():
        if np.iscomplexobj(np.asarray(v)):
            hits.append((name, k, np.asarray(v).ravel()[0]))
    # returning non-None would halt visititems

store, path = registry.resolve(items[0][1])
with h5py.File(BlockStoreReader(store, path), "r") as f:
    f.visititems(collect_complex)
print(f"{len(hits)} complex-valued attributes in this granule, e.g.\n")
for n, k, v in hits[:4]:
    print(f"  /{n}\n     {k} = {v!r}")

NISAR's calibration group stores crosstalk statistics as complex numbers. Zarr v3 metadata is
JSON, and JSON has no complex type, so serializing that group's `zarr.json` raises
`TypeError: Object of type complex is not JSON serializable`.

**This is not a blocker, it is a constraint on usage:** scope the parser to
`/science/LSAR/<PRODUCT>/grids/frequencyA` and the problem disappears. But it does mean a virtual
NISAR cube carries the imagery *without* the calibration and processing metadata that lives
elsewhere in the file — you have to fetch that separately if a workflow needs it.

Parts of the metadata tree do virtualize; others hit a second issue (variables of different
lengths colliding on the same auto-generated `phony_dim_0`):

In [ ]:
for grp in ("/science/LSAR/GCOV/metadata/sourceData",
            "/science/LSAR/identification"):
    try:
        v = vz.open_virtual_dataset(items[0][1], registry=registry,
                                    parser=HDFParser(group=grp))
        print(f"OK    {grp}: {len(v.variables)} variables")
    except Exception as e:
        print(f"FAIL  {grp}: {type(e).__name__}: {str(e)[:110]}")

## 5. Failure: one track does not mean one grid

Now the real test — concatenating the stack. Build every manifest first.

In [ ]:
t0 = time.perf_counter()
vdss = [vz.open_virtual_dataset(u, registry=registry, parser=parser) for _, u in items]
t_build = time.perf_counter() - t0
print(f"{len(vdss)} manifests in {t_build:.1f}s ({t_build / len(vdss):.2f}s each)")

In [ ]:
try:
    xr.concat(vdss, dim="time", coords="minimal", compat="override", join="exact")
except Exception as e:
    print(f"{type(e).__name__}: {e}")

All 29 granules are the same relative orbit and the same frame, and they still do not share a
grid. Grouping by the actual coordinate arrays shows why — and the virtual coordinates are already
in memory, so this check is free:

In [ ]:
by_grid = defaultdict(list)
for (start, url), v in zip(items, vdss):
    sig = (v.sizes["yCoordinates"], v.sizes["xCoordinates"],
           float(v.xCoordinates[0]), float(v.yCoordinates[0]))
    by_grid[sig].append((start, url, v))

for sig, members in sorted(by_grid.items(), key=lambda kv: -len(kv[1])):
    ny, nx, x0, y0 = sig
    print(f"n={len(members):2d}  {ny} x {nx}  origin=({x0:,.0f}, {y0:,.0f})  "
          f"{members[0][0][:8]} -> {members[-1][0][:8]}")

Two grids: 24 granules on one, and 5 acquisitions from June 2026 onward on a grid shifted ~162 km
east. Same projection (EPSG:32610), same spacing, different extent — a processing-side frame
change partway through the archive.

**This is the most important practical finding in this notebook.** A virtual cube cannot span it.
VirtualiZarr does no reprojection or resampling whatsoever; it only records where bytes live. So
the grouping key for a NISAR virtual cube is not "track and frame" but *the actual grid*, and that
has to be checked, not assumed.

In [ ]:
sig, members = max(by_grid.items(), key=lambda kv: len(kv[1]))
items_g = [(s, u) for s, u, _ in members]
vdss_g = [v for _, _, v in members]
times = pd.to_datetime([s for s, _ in items_g], format="%Y%m%dT%H%M%S")

cube = xr.concat(vdss_g, dim="time", coords="minimal", compat="override", join="exact")
cube = cube.assign_coords(time=("time", times))

nrefs = sum(len(cube[n].data.manifest) for n in cube.data_vars
            if hasattr(cube[n].data, "manifest"))
print(f"{len(vdss_g)} granules -> {cube.nbytes / 1e12:.2f} TB logical, {nrefs:,} chunk references")
cube

## 6. Persist the index with Icechunk

715k chunk references is too many for a Kerchunk JSON sidecar. [Icechunk](https://icechunk.io)
stores them in a transactional, versioned format instead.

NISAR's S3 credentials expire hourly, so the repository is configured with a *refreshable*
credential callback rather than a static token — otherwise the cube stops reading after an hour.

In [ ]:
import shutil
from pathlib import Path
import icechunk

REPO = Path("/tmp/nisar_icechunk"); shutil.rmtree(REPO, ignore_errors=True)
prefix = f"s3://{nv.BUCKET}/"

cfg = icechunk.RepositoryConfig.default()
cfg.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(prefix, icechunk.s3_store(region=nv.REGION))
)
repo = icechunk.Repository.create(
    storage=icechunk.local_filesystem_storage(str(REPO)),
    config=cfg,
    authorize_virtual_chunk_access={
        # nv.icechunk_credentials re-fetches EDL tokens on expiry; it must be a
        # module-level function so icechunk can pickle it into the config.
        prefix: icechunk.Credentials.S3(
            icechunk.s3_refreshable_credentials(nv.icechunk_credentials)
        )
    },
)

t0 = time.perf_counter()
session = repo.writable_session("main")
cube.vz.to_icechunk(session.store)
snapshot = session.commit(f"NISAR GCOV virtual cube, track {key}")
t_write = time.perf_counter() - t0

on_disk = sum(f.stat().st_size for f in REPO.rglob("*") if f.is_file())
print(f"committed {snapshot} in {t_write:.1f}s")
print(f"index on disk : {on_disk / 1e6:5.1f} MB")
print(f"pixels indexed: {cube.nbytes / 1e12:5.2f} TB  "
      f"({cube.nbytes / on_disk:,.0f}x)")

## 7. Re-open — this is the payoff

The 25 s manifest build was a one-time cost. Anyone who can reach this repository now opens the
whole 24-date archive as a single Zarr store:

In [ ]:
t0 = time.perf_counter()
back = xr.open_zarr(repo.readonly_session("main").store,
                    consolidated=False, zarr_format=3, chunks={})
t_open = time.perf_counter() - t0
print(f"re-open: {t_open:.2f}s   (vs {t_build:.0f}s to rebuild, "
      f"{t_baseline * len(vdss_g):.0f}s for the h5netcdf route)")
back

## 8. Correctness: are the pixels actually right?

A virtual store that returns *plausible* numbers would be worse than useless. Read a window
straight out of the source HDF5 with h5py and compare.

In [ ]:
ys, xs = slice(20000, 20016), slice(15000, 15016)
for gi in (0, len(items_g) // 2, len(items_g) - 1):
    store, path = registry.resolve(items_g[gi][1])
    with h5py.File(BlockStoreReader(store, path), "r") as f:
        truth = f[f"{nv.GCOV_GRIDS}/HHHH"][ys, xs]
    got = back["HHHH"].isel(time=gi, yCoordinates=ys, xCoordinates=xs).values
    print(f"{items_g[gi][0][:8]}  identical={np.array_equal(truth, got, equal_nan=True)}  "
          f"mean={np.nanmean(truth):.5f}")

Byte-identical, as it must be — Zarr is decoding the exact same compressed chunks the HDF5
library would have.

## 9. A real query through the virtual cube

Mt. Rainier backscatter over the 24 available dates, straight out of the index:

In [ ]:
t0 = time.perf_counter()
aoi = back["HHHH"].sel(xCoordinates=slice(596_000, 606_000),
                       yCoordinates=slice(5_197_000, 5_187_000))
series = aoi.mean(dim=("yCoordinates", "xCoordinates")).compute()
print(f"{aoi.shape} -> {time.perf_counter() - t0:.1f}s "
      f"({aoi.nbytes / 1e6:.0f} MB streamed from the original .h5 files)")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(series.time, series, "o-", color="#1f77b4")
ax.set_ylabel("mean HHHH ($\\gamma^0$)")
ax.set_title("Mt. Rainier backscatter, NISAR GCOV — read through a virtual Zarr cube")
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()

## 10. Which NISAR products can be virtualized?

GCOV was the easy case. Checking the others:

In [ ]:
import earthaccess

def first_url(short_name):
    for g in earthaccess.search_data(short_name=short_name, bounding_box=BBOX, count=5):
        for u in g.data_links(access="direct"):
            if u.endswith(".h5"):
                return u

for short_name, grp in (("NISAR_L2_GSLC_PROVISIONAL_V1", "/science/LSAR/GSLC/grids/frequencyA"),
                        ("NISAR_L1_RSLC_PROVISIONAL_V1", "/science/LSAR/RSLC/swaths/frequencyA")):
    u = first_url(short_name)
    try:
        v = vz.open_virtual_dataset(u, registry=registry, parser=HDFParser(group=grp))
        print(f"OK    {short_name}: {list(v.data_vars)[:4]}")
    except Exception as e:
        print(f"FAIL  {short_name}: {type(e).__name__}: {str(e)[:130]}")

GSLC works unchanged — its `complex64` maps to a native Zarr v3 dtype.

RSLC fails, and the reason is structural rather than about dtypes. GCOV and GSLC attach HDF5
**dimension scales** to their rasters, giving real coordinate names; RSLC's swath rasters attach
none, so every dimension is auto-named `phony_dim_N` positionally and 1-D variables of different
lengths collide on `phony_dim_0`:

In [ ]:
for short_name, grp, var in (("NISAR_L2_GCOV_PROVISIONAL_V1", nv.GCOV_GRIDS, "HHHH"),
                             ("NISAR_L2_GSLC_PROVISIONAL_V1", "/science/LSAR/GSLC/grids/frequencyA", "HH"),
                             ("NISAR_L1_RSLC_PROVISIONAL_V1", "/science/LSAR/RSLC/swaths/frequencyA", "HH")):
    store, path = registry.resolve(first_url(short_name))
    with h5py.File(BlockStoreReader(store, path), "r") as f:
        d = f[f"{grp}/{var}"]
        scales = [[s.name.split("/")[-1] for s in d.dims[i].values()] for i in range(d.ndim)]
    print(f"{short_name.split('_')[2]:5s} {var:5s} dimension scales: {scales}")

Dropping the non-raster variables works around it:

In [ ]:
u = first_url("NISAR_L1_RSLC_PROVISIONAL_V1")
grp = "/science/LSAR/RSLC/swaths/frequencyA"
store, path = registry.resolve(u)
with h5py.File(BlockStoreReader(store, path), "r") as f:
    drop = [k for k, v in f[grp].items()
            if isinstance(v, h5py.Dataset) and not (v.ndim == 2 and v.shape[1] != 2)]

rslc = vz.open_virtual_dataset(u, registry=registry,
                               parser=HDFParser(group=grp, drop_variables=drop))
print(rslc)
print("\nHH dtype :", rslc["HH"].data.metadata.data_type)
print("HH codecs:", rslc["HH"].data.metadata.codecs)

So RSLC *is* virtualizable, but the result has no geolocation attached — `phony_dim_0/1` instead
of coordinates. For pixel-tracking workflows that read RSLC in radar geometry that may be fine;
for anything that needs geocoding it is a real handicap, and the radar grid would have to be
reattached from the metadata tree that section 4 showed we cannot include in the same store.

(Worth noting: these PROVISIONAL RSLC products store `complex64`, not the `CFloat16` the mission
documentation describes for RSLC. Had they been `CFloat16` — an HDF5 compound of two float16 —
there is no corresponding Zarr v3 dtype and this would have failed outright. If NISAR ships
CFloat16 RSLC later, this conclusion needs re-testing.)

## Conclusions

**VirtualiZarr works on NISAR data, and the payoff is specific: it eliminates the metadata cost
of opening an archive, repeatedly, for everyone.** Building the index for 24 granules took 25 s
once; re-opening the resulting 0.56 TB cube takes 0.05 s, from a 13 MB file, with pixels
byte-identical to the source HDF5. Nothing is copied and nothing is duplicated in S3.

**It is worth doing when** you have a repeat-pass stack on a fixed grid that several people (or
one person, many times) will query — exactly the shape of the snow-melt-timing and glacier
time-series workflows this project targets.

**It is not worth doing when** you want one granule, or one small AOI once. Subsetting and writing
a real Zarr store is simpler and faster for that, because VirtualiZarr never reduces the pixel
bytes you stream — only the metadata you re-read.

**The constraints that matter, all confirmed above:**

- Scope the parser to a `grids/frequencyX` group. Root-level parsing fails on complex-valued
  calibration attributes, so a virtual cube cannot carry NISAR's full metadata tree.
- Group granules by their **actual coordinate arrays**, not by track and frame. We found two
  incompatible grids inside a single relative orbit and frame.
- VirtualiZarr cannot reproject, resample, or regrid. Mixed UTM zones or shifted frames need one
  cube each, or a genuine (data-copying) regrid.
- GCOV and GSLC work as-is. RSLC needs `drop_variables` and loses geolocation.
- Use Icechunk, not Kerchunk JSON — 715k references per cube.
- Refresh S3 credentials: EDL tokens last an hour, and a static token silently expires mid-cube.

**Worth trying next:** parallelizing the manifest build with `open_virtual_mfdataset`, appending
new acquisitions to the Icechunk repo as they are published (the transactional model supports it),
and putting the repo in S3 so the index is shared rather than rebuilt per user.